# 03 — Preprocessing & Feature Engineering

**Input:** `reduced_dataset.csv` (via `RAW_PATH` in config)  
**Output:** `cleaned_dataset.csv` and `features_listing.csv`  
**Target:** `sold_price`

**Notebook covers:**
1. Load data and reconcile sold_price columns
2. homeData column recovery — fill nulls from scraped source
3. listing_price string → numeric conversion
4. Drop irrelevant columns and filter bad rows
5. Parse lot_size_sqft from dimension strings
6. Impute remaining nulls
7. Feature engineering — encode, derive new features, OHE
8. Temporal feature — week_of_month from days_since_sold
9. VIF drop + final feature selection + save

## Setup

Import libraries, load config, and read the raw dataset into `raw_df`. A copy is kept as `df` for all transformations — `raw_df` is preserved untouched for the health check at the end.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.stats import f_oneway
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

%run config.ipynb

raw_df = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Shape: {raw_df.shape}')
raw_df.head(3)

Shape: (34652, 68)


,homeData_propertyId,homeData_listingId,homeData_mlsId,homeData_url,homeData_mlsStatusId,homeData_propertyType,homeData_priceInfo_amount,homeData_priceInfo_displayLevel,homeData_priceInfo_priceType,homeData_priceInfo_homePrice_displayLevel,...,lot_size,taxes,walk_score,zolo_estimate,source,median_income,population,parking_spaces,listing_price,region
0,152867555.0,210396185.0,C12652918,/on/toronto/1741-Bayview-Ave-M4G-3C5/home/1528...,2540.0,4.0,6900000.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6900000.0,TORONTO
1,192254240.0,213574315.0,N12944138,/on/toronto/3-McCowan-Rd-M1M-1P1/home/192254240,4156.0,8.0,339900.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,459000.0,TORONTO
2,151883841.0,212786096.0,E12854246,/on/toronto/3050-Ellesmere-Rd-M1E-5E6/unit-110...,4156.0,3.0,459000.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,474000.0,TORONTO


## 1. Reconcile sold_price and Rename Columns

The raw merged dataset contains two sold price columns (`sold_price_x` and `sold_price_y`) from the two data sources. These are unified using `combine_first` — `sold_price_x` takes priority since it has 0 nulls.

Column renames are applied using `UPDATED_NAME_COLS` from config (`square_feet → square_feet_num`, `lot_size → lot_size_sqft`).

In [2]:
df = raw_df.copy()

df['sold_price'] = df[DIFFERENT_SOLD_PRICES_COLS[0]].combine_first(df[DIFFERENT_SOLD_PRICES_COLS[1]])
df = df.drop(columns = list(DIFFERENT_SOLD_PRICES_COLS))

df = df.rename(columns = UPDATED_NAME_COLS)

checking_function(df, pretext = '03_preprocessing.ipynb', pre_check = False)
print(f'shape after consiladation{df.shape}')
print(f'sold prices nulls {df['sold_price'].isnull().sum()}')

[03_preprocessing.ipynb] Schema Ok shape (34652, 67)
shape after consiladation(34652, 67)
sold prices nulls 0


## 2. homeData Column Recovery

The dataset has two sources — TRREB rows (clean columns populated) and scraped rows (clean columns null, homeData columns populated). This step fills nulls in the clean columns using their `homeData_*` equivalents as defined in `FILL_HOMEDATA_COLS` in config.

**Columns recovered:** `bedrooms`, `bathrooms`, `city`, `latitude`, `longitude`, `lot_size_sqft`

In [3]:
for datahome_col, col_clean in FILL_HOMEDATA_COLS.items():
    if datahome_col in df.columns and col_clean in df.columns:
        before_null = df[col_clean].isnull().sum()
        df[col_clean] = df[col_clean].fillna(df[datahome_col])
        after_null = df[col_clean].isnull().sum()
        print(f"{col_clean}: {before_null} nulls {after_null} nulls (recovered {before_null - after_null })")

bedrooms: 22635 nulls 371 nulls (recovered 22264)
bathrooms: 22595 nulls 305 nulls (recovered 22290)
city: 22563 nulls 0 nulls (recovered 22563)
latitude: 22563 nulls 0 nulls (recovered 22563)
longitude: 22563 nulls 0 nulls (recovered 22563)
lot_size_sqft: 24336 nulls 0 nulls (recovered 24336)


### 2.1 Missingness Audit After Recovery

Confirming null percentages on key columns after the homeData fill. `bedrooms`, `bathrooms`, `city`, `latitude`, `longitude` should all be at or near 0%.

In [4]:
cols_audit = [i for i in CHECK_COL if i in df.columns]
print("Missing % after homeData recovery")
print((df[cols_audit].isnull().mean()* 100)
      .sort_values(ascending=False).round(1).to_string())

Missing % after homeData recovery
parking_spaces     65.3
taxes              65.2
house_age          65.2
property_type      65.1
median_income      65.1
square_feet_num    65.1
population         65.1
listing_price      64.8
bedrooms            1.1
bathrooms           0.9
sold_price          0.0
longitude           0.0
latitude            0.0
city                0.0
lot_size_sqft       0.0


## 3. listing_price — String to Numeric Conversion

`listing_price` comes in as a string dtype in this dataset (e.g. `'499995.0'`). Converting to float using a lambda that strips any `$` and `,` characters before casting.

Also diagnosing rows with suspiciously low listing prices — any rows below `MIN_LISTING_PRICE` (from config) will be removed in Step 4.

In [5]:
df['listing_price'] = df['listing_price'].apply(
    lambda x: float(x.replace('$', '').replace(",","")) 
    if isinstance(x, str) else x).astype(float)
print(f'dtype {df['listing_price'].dtype}')
print(f'nulls {df['listing_price'].isnull().sum()}')
print(f'range ${df['listing_price'].min():,.0f} - ${df['listing_price'].max():,.0f}')

dtype float64
nulls 22451
range $1 - $28,650,000


### 3.1 Low Listing Price Diagnostic

Checking how many rows fall below the $1,000 and $10,000 thresholds and inspecting what they look like. These are data errors — no residential listing in the GTA has a genuine listing price of $1.

In [6]:
print(f'rows with listing price {(df['listing_price'] < 1000).sum()}')
print(f'rows with listing price {(df['listing_price']< 10000).sum()}')
print(df[df['listing_price'] < 1000][["listing_price", 'sold_price', 'city', 'property_type']].head(10))

rows with listing price 2
rows with listing price 2
       listing_price  sold_price    city property_type
15625            1.0    748000.0    Peel      Detached
24606            1.0    820000.0  Durham      Detached


## 4. Drop Columns and Filter Rows

Three cleanup steps in sequence:

1. **Drop homeData columns** — all `homeData_*` columns except those used for recovery (`HOMEDATA_REMAINDER`) are dropped
2. **Drop irrelevant columns** — IDs, free text, date columns, 100% missing columns as defined in `COL_TO_DROP` in config
3. **Filter bad rows** — listing prices below `MIN_LISTING_PRICE`, 0-bedroom listings replaced with 1 (`FILL_BEDROOMS`), and rows with nulls in critical columns (`ROW_NULL_DROP`) dropped

In [7]:

datahome_drop = [i for i in df.columns
                if i.startswith('homeData_') and i not in HOMEDATA_REMAINDER]
df = df.drop(columns = datahome_drop)

df = df.drop(columns = [i for i in COL_TO_DROP if i in df.columns])

before_filter = len(df)
df = df[df["listing_price"] >= MIN_LISTING_PRICE]
print(f'Rows removed (listing price < {MIN_LISTING_PRICE}): {before_filter - len(df)}')

df['bedrooms'] = df['bedrooms'].replace(0, FILL_BEDROOMS)

before_filter = len(df)
df = df.dropna(subset=[i for i in ROW_NULL_DROP if i in df.columns])
print(f"Rows removed (null in critial cols): {before_filter - len(df)}")

print(f"\n Shape after drops: {df.shape}")
print(f"Remaining columsn: {df.columns.tolist()}")

Rows removed (listing price < 1000): 22453
Rows removed (null in critial cols): 152

 Shape after drops: (12047, 25)
Remaining columsn: ['homeData_lotSize_amount', 'homeData_addressInfo_centroid_centroid_latitude', 'homeData_addressInfo_centroid_centroid_longitude', 'homeData_addressInfo_city', 'homeData_beds', 'homeData_baths', 'sold_date_dt', 'days_since_sold', 'city', 'bedrooms', 'bathrooms', 'square_feet_num', 'number_of_storeys', 'latitude', 'longitude', 'property_type', 'house_age', 'lot_size_sqft', 'taxes', 'median_income', 'population', 'parking_spaces', 'listing_price', 'region', 'sold_price']


## 5. Parse lot_size_sqft

`lot_size_sqft` comes in as a dimension string in this dataset (e.g. `'116.73 x 197.92 Feet'`, `'7.29 x 7.29 Acres'`). This step parses the two dimensions, multiplies them to get area, and converts acres to square feet where needed (1 acre = 43,560 sqft).

Must run **before** imputation so the median is calculated on numeric values, not strings.

In [8]:
def lot_size_parse(values):
    if pd.isna(values):
        return np.nan
    if isinstance(values, (int, float)):
        return float(values)
    values = str(values).strip()

    try:
        part = values.lower().replace('feet', '').replace('acres', '').strip().split('x')
        if len(part) == 2:
            dims1 = float(part[0].strip())
            dims2 = float(part[1].strip())
            areas = dims1 * dims2

            if 'acre' in values.lower():
                areas = areas * 43560
            return areas
    except:
        return np.nan
    return np.nan

df['lot_size_sqft'] = df['lot_size_sqft'].apply(lot_size_parse)

## 6. Impute Remaining Nulls

Two imputation strategies:
- **Numeric columns** — filled with column median (robust to outliers)
- **Categorical columns** — filled with column mode (most frequent value)

Columns imputed are defined inline rather than in config since they're specific to what remains after the drop step.

In [9]:
impute_numeric = ['bedrooms', 'bathrooms', 'parking_spaces', 'taxes',
                 'median_income', 'population', 'number_of_storeys', 'days_since_sold']

for cols in impute_numeric:
    if cols in df.columns and df[cols].isnull().sum() > 0:
        val_median = df[cols].median()
        df[cols] = df[cols].fillna(val_median)
        print(f'{cols}: imputed with median {val_median:.2f}')

for cols in CATEGORICAL_COLS:
    if cols in df.columns and df[cols].isnull().sum() > 0:
        val_mode = df[cols].mode()[0]
        df[cols] = df[cols].fillna(val_mode)
        print(f'{cols}: imputed with mode "{val_mode}"')    

    
print(f"\n total nulls remaining: {df.isnull().sum().sum()}")

parking_spaces: imputed with median 4.00
taxes: imputed with median 5314.00
number_of_storeys: imputed with median 2.00

 total nulls remaining: 1404


## 7. Feature Engineering

Building derived features and encoding categorical columns:

- **Normalize** — fix `property_type` typo variants using `NORMLIZE_PROPERTY_TYPE` from config
- **Ordinal encode** — `house_age` mapped to numeric order using `NEW_AGE_ORDER` from config
- **Derived numeric features** — `price_per_sqft`, `price_diff_pct`, `beds_plus_baths`, `taxes_per_sqft`, `bath_bed_ratio`
- **Bucket property_type** — top `N_PROPERTY_TYPE_TOP` types kept, rest → `'Others'`
- **One-hot encode** — `property_type` and `city` using `ONE_HOT_COLS` from config

In [10]:
df['property_type'] = df['property_type'].replace(NORMLIZE_PROPERTY_TYPE)
df["encoded_house_age"] = df['house_age'].map(NEW_AGE_ORDER)
df =df.drop(columns=['house_age'])

if 'listing_date' not in df.columns:
    pass
df['listing_price'] = pd.to_numeric(df['listing_price'], errors = 'coerce')
df['price_per_sqft'] = df['listing_price'] / df['lot_size_sqft'].replace(0, np.nan)
df['price_diff_pct'] = (df['sold_price'] - df['listing_price']) /df['listing_price'] * 100
df['beds_plus_baths'] = df['bedrooms'] + df['bathrooms']
df['taxes_per_sqft'] = df['taxes'] / df['lot_size_sqft'].replace(0, np.nan)
df['bath_bed_ratio'] = df['bathrooms'] / df['bedrooms']

type_top =df['property_type'].value_counts().nlargest(N_PROPERTY_TYPE_TOP).index
df['property_type'] = df['property_type'].where(df['property_type'].isin(type_top), other = "Others")
df = pd.get_dummies(df, columns = ONE_HOT_COLS, drop_first = True)

print(f"Shape after encoding: {df.shape}")
print(f" Nulls remaining: {df.isnull().sum().sum()}")

Shape after encoding: (12047, 57)
 Nulls remaining: 7582


## 8. Temporal Feature — week_of_month

Converting `days_since_sold` into `week_of_month` (1–4) using a reference date of 2026-04-12. This captures intra-month seasonality in listing activity without leaking future information.

Both `days_since_sold` and the intermediate `sold_date_dt` column are dropped after extraction.

In [11]:
ref_date = pd.Timestamp("2026-04-12", tz="UTC")
df["sold_date_dt"] = ref_date - pd.to_timedelta(df["days_since_sold"], unit="D")
df["week_of_month"] = ((df["sold_date_dt"].dt.day - 1) // 7) + 1

df = df.drop(columns=["days_since_sold", "sold_date_dt"])
print(f'week_of_month sample: \n {df['week_of_month'].value_counts().sort_index().to_string()}')

week_of_month sample: 
 week_of_month
1    3045
2    2580
3    2763
4    2767
5     892


## 9. VIF Drop + Final Feature Selection + Save

Removing high-multicollinearity columns identified in VIF analysis (`DROP_COLS_VIF` from config), then building the final feature set from `FEATURES_FINAL` plus all OHE columns generated in Step 7.

Two files are saved:
- `PROCESSED_PATH` (`cleaned_dataset.csv`) — full cleaned dataset
- `FEATURES_PATH` (`features_listing.csv`) — final feature set ready for modeling

In [12]:
model_df = df.drop(columns = [ i for i in DROP_COLS_VIF if i in df.columns])

encoded_cols = [i for i in model_df.columns
                if i.startswith('property_type_') or i.startswith('city_')]

cols_feature = FEATURES_FINAL + encoded_cols
cols_feature = [i for i in cols_feature if i in model_df.columns]

df_final = model_df[cols_feature + ['sold_price']].dropna()

print(f"final dataset shape: {df_final.shape}")
print(f"Features: {cols_feature}")
print(f"Nulls: {model_df.isnull().sum().sum()}")

df_final.to_csv(FEATURES_PATH, index = False)
df.to_csv(PROCESSED_PATH, index = False)

health_stat(raw_df, df_final, output_path=FEATURES_PATH, label = 'Preprocessing + Feature Enginerring')

final dataset shape: (12017, 35)
Features: ['week_of_month', 'listing_price', 'bathrooms', 'parking_spaces', 'encoded_house_age', 'property_type_Common Element Condo', 'property_type_Condo Apartment', 'property_type_Condo Townhouse', 'property_type_Detached', 'property_type_House', 'property_type_Link', 'property_type_Others', 'property_type_SEMI-DETACHED', 'city_Aurora', 'city_Brampton', 'city_Brock', 'city_Burlington', 'city_Caledon', 'city_Clarington', 'city_Durham', 'city_Halton', 'city_Halton Hills', 'city_King', 'city_Markham', 'city_Mississauga', 'city_Oshawa', 'city_Peel', 'city_Pickering', 'city_Richmond Hill', 'city_Scugog', 'city_Vaughan', 'city_Whitby', 'city_Whitchurch-stouffville', 'city_York']
Nulls: 4493

 Health Check Preprocessing + Feature Enginerring
 Input shape (34652, 68)
 Output shape (12017, 35)
 Rows dropped 22,635  (65.3%)
 Cols dropped 33
 Nulls left  0.00%
 Saved to   : features_listing.csv
